# Fraud Detection Model
Detects potentially fraudulent insurance claims using ML classification

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import joblib
import os

In [ ]:
# Generate synthetic claim data
np.random.seed(42)
n_samples = 5000

# Generate features
data = {
    'claim_amount': np.random.uniform(500, 20000, n_samples),
    'claim_age_days': np.random.exponential(20, n_samples),
    'has_evidence': np.random.choice([0, 1], n_samples, p=[0.3, 0.7]),
    'trigger_confirmed': np.random.choice([0, 1], n_samples, p=[0.4, 0.6]),
    'user_claim_count': np.random.poisson(2, n_samples)
}

df = pd.DataFrame(data)

# Generate fraud labels (ground truth)
def calculate_fraud_label(row):
    fraud_prob = 0.0
    if row['claim_amount'] > 10000:
        fraud_prob += 0.25
    if row['claim_age_days'] > 30:
        fraud_prob += 0.2
    if row['has_evidence'] == 0:
        fraud_prob += 0.3
    if row['trigger_confirmed'] == 0:
        fraud_prob += 0.35
    if row['user_claim_count'] > 3:
        fraud_prob += 0.2
    return 1 if np.random.random() < min(fraud_prob, 1.0) else 0

df['is_fraud'] = df.apply(calculate_fraud_label, axis=1)

print(f"Total claims: {len(df)}")
print(f"Fraudulent claims: {df['is_fraud'].sum()} ({df['is_fraud'].mean()*100:.1f}%)")
df.head()

In [ ]:
# Prepare features and target
feature_cols = ['claim_amount', 'claim_age_days', 'has_evidence', 'trigger_confirmed', 'user_claim_count']
X = df[feature_cols]
y = df['is_fraud']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Fraud rate in train: {y_train.mean()*100:.1f}%")
print(f"Fraud rate in test: {y_test.mean()*100:.1f}%")

In [ ]:
# Train Random Forest classifier
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42
)

model.fit(X_train, y_train)
print("Model trained successfully!")

In [ ]:
# Evaluate model
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print(f"\nROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

In [ ]:
# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("Feature Importance:")
print(importance.to_string(index=False))

In [ ]:
# Save model
os.makedirs('models', exist_ok=True)
model_path = 'models/fraud_model.joblib'
joblib.dump(model, model_path)
print(f"Model saved to {model_path}")

In [ ]:
# Test fraud prediction
test_claims = [
    {'claim_amount': 5000, 'claim_age_days': 10, 'has_evidence': 1, 'trigger_confirmed': 1, 'user_claim_count': 1},
    {'claim_amount': 15000, 'claim_age_days': 45, 'has_evidence': 0, 'trigger_confirmed': 0, 'user_claim_count': 5},
]

for claim in test_claims:
    feature_array = np.array([list(claim.values())])
    fraud_prob = model.predict_proba(feature_array)[0][1]
    print(f"Claim: {claim['claim_amount']}, Evidence: {claim['has_evidence']}, Trigger: {claim['trigger_confirmed']}")
    print(f"  Fraud Probability: {fraud_prob*100:.1f}%")